# 02 — Dense, kNN and Fusion Search

**Question.** Which dense model, chunking, calibration-query kNN, and RRF weights improve the frozen BM25F baseline?

All selection remains tune-only; the chosen hybrid is evaluated once on confirm.


## Setup


In [ ]:
from pathlib import Path
import os, sys, json, time, copy, itertools, importlib.util, subprocess

def locate_project(start=Path.cwd()):
    for parent in [start.resolve(), *start.resolve().parents]:
        if (parent / "pyproject.toml").exists() and (parent / "src/avito_retriever").exists():
            return parent
    candidate = Path("/content/avito-retriever")
    if candidate.exists():
        return candidate
    raise FileNotFoundError("Clone/open avito-retriever or change PROJECT_ROOT in this cell")

PROJECT_ROOT = locate_project()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

def module_available(name):
    try:
        return importlib.util.find_spec(name) is not None
    except ModuleNotFoundError:
        return False

IN_COLAB = module_available("google.colab")
if IN_COLAB and os.environ.get("AVITO_AUTO_INSTALL", "1") != "0":
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-e",
        f"{PROJECT_ROOT}[lexical,neural,dev]",
    ])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from avito_retriever.experiments.notebook import resolve_data_dir, result_dir, highlight_best, save_json, save_yaml

DATA_DIR = resolve_data_dir(PROJECT_ROOT)
SEED = 42
np.random.seed(SEED)
print("Project:", PROJECT_ROOT)
print("Data:", DATA_DIR)


In [ ]:
import yaml
from avito_retriever.config import load_config
from avito_retriever.data.io import load_calibration
from avito_retriever.retrieval.dense import DenseIndex, build_chunks
from avito_retriever.retrieval.knn import lexical_neighbours_oof, dense_neighbours_oof, fuse_neighbours_rrf, neighbours_to_article_rankings
from avito_retriever.evaluation.cv import make_grouped_query_folds
from avito_retriever.evaluation.metrics import evaluate_rankings
from avito_retriever.evaluation.selection import metric_on_split
from avito_retriever.fusion.rrf import weighted_rrf
from avito_retriever.tokenization.sentencepiece import SentencePieceTokenizer

OUT = result_dir(PROJECT_ROOT, "02_dense_knn_fusion_search")
PREV = result_dir(PROJECT_ROOT, "01_sentencepiece_bm25f_search")
parsed = pd.read_parquet(PROJECT_ROOT / "artifacts/cache/parsed_articles.parquet")
calibration = load_calibration(DATA_DIR)
split = pd.read_csv(PREV / "selection_split.csv")
folds = make_grouped_query_folds(calibration, n_folds=5, seed=SEED)
bm25f_ranking = pd.read_parquet(PREV / "best_bm25f_rankings.parquet")
sp_model = json.loads((PREV / "best_artifacts.json").read_text())["sentencepiece_model"]
sp = SentencePieceTokenizer(sp_model)
base = load_config(PROJECT_ROOT / "configs/experiments/hybrid_knn.yaml")


## Dense model and chunk search space


In [ ]:
DENSE_MODELS = ["intfloat/multilingual-e5-small", "deepvk/USER2-base", "deepvk/USER-bge-m3"]
CHUNKS = [(128, 32), (256, 48), (384, 64)]
BATCH_SIZE = 64
print(f"Dense trials: {len(DENSE_MODELS)*len(CHUNKS)}; GPU recommended")


## Dense trials


In [ ]:
progress_path = OUT / "dense_leaderboard.csv"
dense_rows = pd.read_csv(progress_path).to_dict("records") if progress_path.exists() else []
completed = {row["trial_id"] for row in dense_rows if pd.notna(row.get("tune_map@10"))}
dense_files = {}
for model_name, (size, overlap) in itertools.product(DENSE_MODELS, CHUNKS):
    trial = f"{model_name.split('/')[-1]}-{size}-{overlap}"
    path = OUT / f"dense_{trial}.parquet"; dense_files[trial] = str(path)
    if trial in completed and path.exists(): continue
    started = time.perf_counter()
    try:
        chunks = build_chunks(parsed, size_words=size, overlap_words=overlap)
        query_frame = calibration[["query_id", "query_text"]].copy()
        if "e5" in model_name.lower():
            chunks["text"] = "passage: " + chunks.text
            query_frame["query_text"] = "query: " + query_frame.query_text
        cfg = {**base["retrieval"]["dense"], "model_name": model_name, "batch_size": BATCH_SIZE}
        dense = DenseIndex(cfg, PROJECT_ROOT / "artifacts/embeddings" / trial)
        dense.fit_chunks(chunks)
        ranking = dense.retrieve(query_frame, top_k_articles=100)
        metrics, per_query = evaluate_rankings(ranking, calibration)
        ranking.to_parquet(path, index=False)
        per_query.to_parquet(OUT / f"dense_per_query_{trial}.parquet", index=False)
        dense_rows.append({"trial_id": trial, "model_name": model_name, "chunk_size": size,
                           "overlap": overlap, "tune_map@10": metric_on_split(per_query, split, "ap@10", "tune"),
                           "seconds": time.perf_counter()-started, "status": "ok", **metrics})
    except Exception as error:
        dense_rows.append({"trial_id": trial, "model_name": model_name, "chunk_size": size,
                           "overlap": overlap, "seconds": time.perf_counter()-started,
                           "status": f"error: {error}"})
    pd.DataFrame(dense_rows).drop_duplicates("trial_id", keep="last").to_csv(progress_path, index=False)
dense_results = pd.DataFrame(dense_rows).drop_duplicates("trial_id", keep="last")
dense_table = dense_results.dropna(subset=["tune_map@10"]).sort_values("tune_map@10", ascending=False).reset_index(drop=True)
assert not dense_table.empty, "Every dense trial failed; inspect dense_leaderboard.csv"
display(highlight_best(dense_table, ["tune_map@10", "map@10", "recall@50", "recall@100"]))


## SentencePiece-BM25 and dense kNN


In [ ]:
best_dense = dense_table.iloc[0].to_dict()
best_dense_ranking = pd.read_parquet(dense_files[best_dense["trial_id"]])
_, best_dense_per_query = evaluate_rankings(best_dense_ranking, calibration)
best_dense_ranking.to_parquet(OUT / "best_dense_rankings.parquet", index=False)
best_dense_per_query.to_parquet(OUT / "best_dense_per_query.parquet", index=False)
lex_cfg = base["retrieval"]["knn"]["lexical"]
lex_neighbours = lexical_neighbours_oof(calibration, folds, sp.encode, lex_cfg["k1"], lex_cfg["b"], depth=30)

# Reuse query embeddings from the selected dense model.
chunks = build_chunks(parsed, int(best_dense["chunk_size"]), int(best_dense["overlap"]))
query_frame = calibration[["query_id", "query_text"]].copy()
if "e5" in best_dense["model_name"].lower():
    chunks["text"] = "passage: " + chunks.text; query_frame["query_text"] = "query: " + query_frame.query_text
dense = DenseIndex({**base["retrieval"]["dense"], "model_name": best_dense["model_name"], "batch_size": BATCH_SIZE},
                   PROJECT_ROOT / "artifacts/embeddings" / best_dense["trial_id"])
dense.fit_chunks(chunks)
query_embeddings = dense.query_embeddings(query_frame)
dense_neighbours = dense_neighbours_oof(calibration, folds, query_embeddings, depth=30)


In [ ]:
knn_rows, knn_rankings = [], {}
for k, lexical_weight, dense_weight in itertools.product([3,5,10,20], [0.5,1.0], [0.5,1.0]):
    trial = f"k{k}-l{lexical_weight}-d{dense_weight}"
    neighbours = fuse_neighbours_rrf({"lexical": lex_neighbours, "dense": dense_neighbours},
                                     {"lexical": lexical_weight, "dense": dense_weight}, rrf_k=20)
    ranking = neighbours_to_article_rankings(neighbours, calibration, top_k_neighbours=k, top_k_articles=100)
    metrics, per_query = evaluate_rankings(ranking, calibration)
    knn_rankings[trial] = ranking
    knn_rows.append({"trial_id": trial, "k": k, "lexical_weight": lexical_weight,
                     "dense_weight": dense_weight, "tune_map@10": metric_on_split(per_query, split, "ap@10", "tune"), **metrics})
knn_table = pd.DataFrame(knn_rows).sort_values("tune_map@10", ascending=False).reset_index(drop=True)
display(highlight_best(knn_table.head(15), ["tune_map@10", "map@10", "recall@50"]))


## Freeze the strongest kNN


In [ ]:
best_knn = knn_table.iloc[0].to_dict()
best_knn_ranking = knn_rankings[best_knn["trial_id"]]
_, best_knn_per_query = evaluate_rankings(best_knn_ranking, calibration)
best_knn_ranking.to_parquet(OUT / "best_knn_rankings.parquet", index=False)
best_knn_per_query.to_parquet(OUT / "best_knn_per_query.parquet", index=False)


## All BM25F / dense / kNN combinations

There are seven non-empty subsets of three retrievers. Single-component systems are already frozen. Each pair and the three-way hybrid receives its own tune-only RRF weight search.


In [ ]:
component_rankings = {
    "bm25f": bm25f_ranking,
    "dense": best_dense_ranking,
    "knn": best_knn_ranking,
}
WEIGHT_GRID = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0]
RRF_K_GRID = [20, 40, 60]
SUBSETS = [
    ("bm25f_dense", ["bm25f", "dense"]),
    ("bm25f_knn", ["bm25f", "knn"]),
    ("dense_knn", ["dense", "knn"]),
    ("bm25f_dense_knn", ["bm25f", "dense", "knn"]),
]

fusion_rows, fusion_rankings = [], {}
for subset_name, components in SUBSETS:
    # The first component is fixed at 1.0 because RRF weights are scale-invariant.
    free_components = components[1:]
    for free_weights in itertools.product(WEIGHT_GRID, repeat=len(free_components)):
        weights = {components[0]: 1.0, **dict(zip(free_components, free_weights))}
        for rrf_k in RRF_K_GRID:
            trial = subset_name + "-" + "-".join(f"{key}{value}" for key, value in weights.items()) + f"-r{rrf_k}"
            ranking = weighted_rrf({key: component_rankings[key] for key in components},
                                   weights, rrf_k=rrf_k, top_k=100)
            metrics, per_query = evaluate_rankings(ranking, calibration)
            fusion_rankings[trial] = ranking
            fusion_rows.append({"subset": subset_name, "trial_id": trial,
                                **{f"{key}_weight": weights.get(key, 0.0) for key in component_rankings},
                                "rrf_k": rrf_k,
                                "tune_map@10": metric_on_split(per_query, split, "ap@10", "tune"),
                                **metrics})

fusion_table = pd.DataFrame(fusion_rows).sort_values(["subset", "tune_map@10"], ascending=[True, False])
fusion_winners = fusion_table.groupby("subset", as_index=False, sort=False).head(1).reset_index(drop=True)
display(highlight_best(fusion_winners, ["tune_map@10", "map@10", "recall@50", "recall@100"]))


## Freeze every architecture variant and build reranker contexts


In [ ]:
architecture_variants = {}
for row in fusion_winners.to_dict("records"):
    ranking = fusion_rankings[row["trial_id"]]
    _, per_query = evaluate_rankings(ranking, calibration)
    ranking.to_parquet(OUT / f"best_{row['subset']}_rankings.parquet", index=False)
    per_query.to_parquet(OUT / f"best_{row['subset']}_per_query.parquet", index=False)
    architecture_variants[row["subset"]] = {
        "trial_id": row["trial_id"],
        "bm25f_weight": float(row["bm25f_weight"]),
        "dense_weight": float(row["dense_weight"]),
        "knn_weight": float(row["knn_weight"]),
        "rrf_k": int(row["rrf_k"]),
        "tune_map@10": float(row["tune_map@10"]),
    }

best_fusion = architecture_variants["bm25f_dense_knn"]
hybrid = pd.read_parquet(OUT / "best_bm25f_dense_knn_rankings.parquet")
hybrid_per_query = pd.read_parquet(OUT / "best_bm25f_dense_knn_per_query.parquet")
hybrid.to_parquet(OUT / "best_hybrid_rankings.parquet", index=False)
hybrid_per_query.to_parquet(OUT / "best_hybrid_per_query.parquet", index=False)
candidates = hybrid[hybrid["rank"] <= 50].copy()
contexts = dense.best_chunk_texts(query_frame, candidates, n_chunks=2)
pd.DataFrame([{"query_id": q, "article_id": a, "text": text} for (q,a),text in contexts.items()]).to_parquet(
    OUT / "reranker_contexts.parquet", index=False)
selected = {"dense": best_dense, "knn": best_knn, "fusion": best_fusion,
            "architecture_variants": architecture_variants,
            "confirm_map@10": metric_on_split(hybrid_per_query, split, "ap@10", "confirm")}
save_json(selected, OUT / "best_hybrid_config.json")
dense_results.to_csv(OUT / "dense_leaderboard.csv", index=False)
knn_table.to_csv(OUT / "knn_leaderboard.csv", index=False); fusion_table.to_csv(OUT / "fusion_leaderboard.csv", index=False)
display(pd.DataFrame([selected["fusion"] | {"confirm_map@10": selected["confirm_map@10"]}]))


## Interpretation
Notebook 03 receives one frozen top-50 candidate pool and identical contexts for every reranker. This makes reranker comparisons paired and fair.
